# 10 · Skill Gap Analysis — IBM JobRole ↔ O*NET Skills

**Project:** Enterprise HR AI  
**Purpose:** For each of the 9 IBM JobRole values, retrieve top essential skills and
top software tools from O*NET via the manually curated `jobrole_onet_mapping.csv`.

> **Confidence caveats are printed alongside every result.**  
> Low-confidence mappings are never presented without their caveat.  
> The `very_low` confidence 'Manager' role receives a placeholder, not fabricated skill data.

---

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

PROC = os.path.join('..', 'data', 'processed')
EXT  = os.path.join('..', 'data', 'external')

# ── Load all data files ──────────────────────────────────────────────────────
mapping   = pd.read_csv(os.path.join(EXT,  'jobrole_onet_mapping.csv'))
occ       = pd.read_csv(os.path.join(PROC, 'occupation_master.csv'))
essential = pd.read_csv(os.path.join(PROC, 'essential_skills_processed.csv'))
software  = pd.read_csv(os.path.join(PROC, 'software_skills_processed.csv'))
employees = pd.read_csv(os.path.join(PROC, 'employee_attrition_processed.csv'))

print('=== FILE SHAPES ===')
print(f'  jobrole_onet_mapping    : {mapping.shape}')
print(f'  occupation_master       : {occ.shape}')
print(f'  essential_skills        : {essential.shape}')
print(f'  software_skills         : {software.shape}')
print(f'  employee_attrition      : {employees.shape}')
print()

# ── Identify exact column names used for ranking ─────────────────────────────
print('=== ESSENTIAL SKILLS: column used for ranking ===')
# Filter to Importance scale only (Scale ID == 'IM') to avoid double-counting
# with Level scale (Scale ID == 'LV'); ranking by 'Data Value'
essential_im = essential[essential['Scale ID'] == 'IM'].copy()
print(f'  Ranking column: "Data Value" (Importance score, Scale ID=="IM")')
print(f'  Skill name column: "Element Name"')
print(f'  After filtering to IM scale: {essential_im.shape[0]:,} rows')
print()

print('=== SOFTWARE SKILLS: ranking approach ===')
# No numeric score in software_skills — software examples are listed flat.
# Ranking by: Hot Technology=='Y' first, then In Demand=='Y', then alphabetical.
# Actual software name is in 'Workplace Example' column.
print('  No numeric score column in software_skills_processed.csv.')
print('  Ranking: Hot Technology=Y first, then In Demand=Y, then alphabetical.')
print('  Software name column: "Workplace Example"')
print('  Category column: "Element Name"')

# ── IBM roles ────────────────────────────────────────────────────────────────
ibm_roles = mapping['ibm_job_role'].tolist()
print()
print(f'IBM job roles in mapping: {ibm_roles}')

=== FILE SHAPES ===
  jobrole_onet_mapping    : (9, 5)
  occupation_master       : (1016, 3)
  essential_skills        : (18200, 15)
  software_skills         : (31821, 7)
  employee_attrition      : (1470, 35)

=== ESSENTIAL SKILLS: column used for ranking ===
  Ranking column: "Data Value" (Importance score, Scale ID=="IM")
  Skill name column: "Element Name"
  After filtering to IM scale: 9,100 rows

=== SOFTWARE SKILLS: ranking approach ===
  No numeric score column in software_skills_processed.csv.
  Ranking: Hot Technology=Y first, then In Demand=Y, then alphabetical.
  Software name column: "Workplace Example"
  Category column: "Element Name"

IBM job roles in mapping: ['Healthcare Representative', 'Human Resources', 'Laboratory Technician', 'Manager', 'Manufacturing Director', 'Research Director', 'Research Scientist', 'Sales Executive', 'Sales Representative']


---
## Per-Role Skill Lookup

For each role: confidence level printed first, then top 10 essential skills
and top 10 software tools. `very_low` confidence Manager gets a placeholder only.


In [2]:
def get_top_essential(soc_code, n=10):
    """Return top-n essential skills by Data Value (Importance) for a SOC code."""
    subset = essential_im[essential_im['O*NET-SOC Code'] == soc_code]
    if subset.empty:
        return []
    top = (subset
           .sort_values('Data Value', ascending=False)
           .drop_duplicates('Element Name')
           .head(n))
    return list(zip(top['Element Name'], top['Data Value'].round(2)))


def get_top_software(soc_code, n=10):
    """Return top-n software tools for a SOC code.
    Priority: Hot Technology=Y, then In Demand=Y, then alphabetical.
    """
    subset = software[software['O*NET-SOC Code'] == soc_code].copy()
    if subset.empty:
        return []
    # Create sort key: hot=0 > in_demand=1 > other=2
    subset['_sort'] = 2
    subset.loc[subset['In Demand'] == 'Y', '_sort'] = 1
    subset.loc[subset['Hot Technology'] == 'Y', '_sort'] = 0
    top = (subset
           .sort_values(['_sort', 'Workplace Example'])
           .drop_duplicates('Workplace Example')
           .head(n))
    return list(zip(top['Workplace Example'], top['Hot Technology'], top['In Demand']))


def fmt_essential(pairs):
    if not pairs:
        return 'No data found for this SOC code.'
    return ' | '.join(f'{name} ({val})' for name, val in pairs)


def fmt_software(triples):
    if not triples:
        return 'No data found for this SOC code.'
    parts = []
    for name, hot, dem in triples:
        tags = []
        if hot == 'Y': tags.append('HOT')
        if dem == 'Y': tags.append('IN-DEMAND')
        tag_str = f' [{"|".join(tags)}]' if tags else ''
        parts.append(f'{name}{tag_str}')
    return ' | '.join(parts)


# Employee count per IBM role
role_counts = employees['JobRole'].value_counts().to_dict()

print('Employee counts per IBM job role:')
for role, count in sorted(role_counts.items()):
    print(f'  {role:<35} {count:>4} employees')

Employee counts per IBM job role:
  Healthcare Representative            131 employees
  Human Resources                       52 employees
  Laboratory Technician                259 employees
  Manager                              102 employees
  Manufacturing Director               145 employees
  Research Director                     80 employees
  Research Scientist                   292 employees
  Sales Executive                      326 employees
  Sales Representative                  83 employees


In [3]:
VERY_LOW_PLACEHOLDER = (
    'Insufficient mapping confidence (very_low) — '
    'use Department-level analysis instead (see Step 12). '
    'O*NET skill data for this role is not reliable.'
)

# ── Main per-role loop ───────────────────────────────────────────────────────
print('=' * 80)
print('PER-ROLE SKILL LOOKUP (with confidence caveats)')
print('=' * 80)

for _, row in mapping.iterrows():
    ibm_role   = row['ibm_job_role']
    soc_code   = row['onet_soc_code']
    onet_title = row['onet_title']
    confidence = row['match_confidence']
    emp_count  = role_counts.get(ibm_role, 0)

    print(f'\n{ibm_role}')
    print(f'  Employees in dataset : {emp_count}')
    print(f'  Mapped O*NET SOC     : {soc_code}  |  {onet_title}')
    print(f'  Match Confidence     : [{confidence.upper()}]')

    if confidence == 'very_low':
        print(f'  Essential Skills     : *** {VERY_LOW_PLACEHOLDER} ***')
        print(f'  Software Tools       : *** {VERY_LOW_PLACEHOLDER} ***')
    else:
        caveat = {
            'medium': '[CAVEAT: approximate mapping — treat as directional guidance]',
            'low':    '[CAVEAT: low-confidence mapping — use as broad reference only; results may not reflect actual role requirements]',
        }.get(confidence, '')
        print(f'  Confidence Note      : {caveat}')

        ess = get_top_essential(soc_code, 10)
        sw  = get_top_software(soc_code, 10)

        print(f'  Top 10 Essential Skills (by Data Value/Importance, Scale ID=IM):')
        if ess:
            for i, (skill, val) in enumerate(ess, 1):
                print(f'    {i:>2}. {skill:<45} {val}')
        else:
            print('    [No essential skills found for this SOC code]')

        print(f'  Top 10 Software Tools (Hot Technology prioritized):')
        if sw:
            for i, (name, hot, dem) in enumerate(sw, 1):
                tags = []
                if hot == 'Y': tags.append('HOT')
                if dem == 'Y': tags.append('IN-DEMAND')
                tag_str = f' [{"|".join(tags)}]' if tags else ''
                print(f'    {i:>2}. {name}{tag_str}')
        else:
            print('    [No software tools found for this SOC code]')

print('\n' + '=' * 80)

PER-ROLE SKILL LOOKUP (with confidence caveats)

Healthcare Representative
  Employees in dataset : 131
  Mapped O*NET SOC     : 41-3091.00  |  Sales Representatives of Services, Except Advertising, Insurance, Financial Services, and Travel
  Match Confidence     : [LOW]
  Confidence Note      : [CAVEAT: low-confidence mapping — use as broad reference only; results may not reflect actual role requirements]
  Top 10 Essential Skills (by Data Value/Importance, Scale ID=IM):
     1. Active Listening                              3.88
     2. Speaking                                      3.88
     3. Reading Comprehension                         3.62
     4. Writing                                       3.38
     5. Active Learning                               3.38
     6. Critical Thinking                             3.38
     7. Monitoring                                    3.12
     8. Mathematics                                   2.62
     9. Learning Strategies                        

---
## Combined Output Table

One row per IBM job role. Top 5 essential skills and top 5 software tools (space-constrained).
Manager row deliberately shows the placeholder note instead of O*NET skill data.


In [4]:
records = []

for _, row in mapping.iterrows():
    ibm_role   = row['ibm_job_role']
    soc_code   = row['onet_soc_code']
    onet_title = row['onet_title']
    confidence = row['match_confidence']
    emp_count  = role_counts.get(ibm_role, 0)

    if confidence == 'very_low':
        top5_essential = VERY_LOW_PLACEHOLDER
        top5_software  = VERY_LOW_PLACEHOLDER
    else:
        ess = get_top_essential(soc_code, 5)
        top5_essential = ' | '.join(f'{s} ({v})' for s, v in ess) if ess else 'N/A'
        sw = get_top_software(soc_code, 5)
        top5_software = ' | '.join(n for n, h, d in sw) if sw else 'N/A'

    records.append({
        'ibm_job_role':       ibm_role,
        'employee_count':     emp_count,
        'onet_soc_code':      soc_code,
        'onet_title':         onet_title,
        'match_confidence':   confidence,
        'top5_essential_skills': top5_essential,
        'top5_software_tools':   top5_software,
    })

combined = pd.DataFrame(records)

# ── Print table summary ───────────────────────────────────────────────────────
print('=== COMBINED TABLE (all 9 roles) ===')
print()
for _, r in combined.iterrows():
    print(f'Role            : {r["ibm_job_role"]}  ({r["employee_count"]} employees)')
    print(f'O*NET SOC       : {r["onet_soc_code"]}  ({r["match_confidence"]} confidence)')
    print(f'Top 5 Skills    : {r["top5_essential_skills"]}')
    print(f'Top 5 Software  : {r["top5_software_tools"]}')
    print()

# ── Confirm Manager row has placeholder ──────────────────────────────────────
print('=== MANAGER ROW VERIFICATION ===')
mgr_row = combined[combined['ibm_job_role'] == 'Manager'].iloc[0]
has_placeholder = VERY_LOW_PLACEHOLDER in mgr_row['top5_essential_skills']
print(f'Manager confidence: {mgr_row["match_confidence"]}')
print(f'Manager essential_skills field: {mgr_row["top5_essential_skills"][:100]}...')
print(f'CONFIRMED placeholder present: {has_placeholder}')
if not has_placeholder:
    raise AssertionError('Manager row must show placeholder, not O*NET skill data!')

=== COMBINED TABLE (all 9 roles) ===

Role            : Healthcare Representative  (131 employees)
O*NET SOC       : 41-3091.00  (low confidence)
Top 5 Skills    : Active Listening (3.88) | Speaking (3.88) | Reading Comprehension (3.62) | Writing (3.38) | Active Learning (3.38)
Top 5 Software  : Apple macOS | HubSpot software | IBM SPSS Statistics | Microsoft Excel | Microsoft Office software

Role            : Human Resources  (52 employees)
O*NET SOC       : 13-1071.00  (medium confidence)
Top 5 Skills    : Speaking (4.12) | Reading Comprehension (4.0) | Active Listening (4.0) | Writing (3.88) | Critical Thinking (3.88)
Top 5 Software  : Adobe Acrobat | Adobe Creative Cloud software | Adobe Illustrator | Adobe InDesign | Adobe Photoshop

Role            : Laboratory Technician  (259 employees)
O*NET SOC       : 29-2012.00  (medium confidence)
Top 5 Skills    : Reading Comprehension (3.75) | Active Listening (3.62) | Critical Thinking (3.25) | Speaking (3.12) | Science (3.0)
Top 5 Sof

In [5]:
out_path = os.path.join(PROC, 'role_skill_profiles.csv')
combined.to_csv(out_path, index=False, encoding='utf-8')
print(f'Saved: {os.path.abspath(out_path)}')
print(f'Shape: {combined.shape}')
print(f'Size : {os.path.getsize(out_path):,} bytes')
print()

# Round-trip verify
verify = pd.read_csv(out_path)
assert len(verify) == 9, f'Expected 9 rows, got {len(verify)}'
assert 'Manager' in verify['ibm_job_role'].values
mgr_check = verify[verify['ibm_job_role'] == 'Manager']['top5_essential_skills'].iloc[0]
assert 'Insufficient mapping confidence' in mgr_check, 'Manager row missing placeholder!'
print('Round-trip verify: OK — 9 rows, Manager row has placeholder.')

# Note on shared SOC code
print()
print('=== SHARED SOC CODE NOTE (logged in data_relationships.md Open Issue #1) ===')
shared = combined.groupby('onet_soc_code')['ibm_job_role'].apply(list)
for soc, roles in shared.items():
    if len(roles) > 1:
        print(f'  SOC {soc} shared by: {roles}')
        print(f'  -> Their top5_essential_skills and top5_software_tools will be identical.')
        print(f'  -> This is a mapping limitation, NOT a modeling error.')
        print(f'  -> Must be labeled as such if surfaced in the Day 4 dashboard.')

Saved: C:\Users\ASUS\Desktop\enterprise_hr_ai\data\processed\role_skill_profiles.csv
Shape: (9, 7)


Size : 2,936 bytes

Round-trip verify: OK — 9 rows, Manager row has placeholder.

=== SHARED SOC CODE NOTE (logged in data_relationships.md Open Issue #1) ===
  SOC 41-3091.00 shared by: ['Healthcare Representative', 'Sales Representative']
  -> Their top5_essential_skills and top5_software_tools will be identical.
  -> This is a mapping limitation, NOT a modeling error.
  -> Must be labeled as such if surfaced in the Day 4 dashboard.
